In [46]:
import json
import os
from pathlib import Path
from tqdm import tqdm

In [47]:
DATA_PATH = Path("../data/raw/LilleLyd")
MANIFEST_PATH = DATA_PATH / "manifest.jsonl"

def load_manifest(manifest_path: Path):
    with manifest_path.open("r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

manifest_lines = list(load_manifest(MANIFEST_PATH))

In [48]:
for entry in manifest_lines:
    entry["audio_filepath"] = entry["audio_filepath"][len("/Users/joachimschroderandersson/Desktop/exp/data/"):]

In [49]:
# Resample and copy to processed directory
PROCESSED_DATA_PATH = Path("../data/processed/LilleLyd")
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

for entry in tqdm(manifest_lines):
    original_path = DATA_PATH / entry["audio_filepath"]
    new_path = PROCESSED_DATA_PATH / entry["audio_filepath"]
    new_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Resample using ffmpeg
    os.system(f"ffmpeg -i '{original_path}' -ar 16000 -ac 1 '{new_path}' -y")

  0%|          | 0/1100 [00:00<?, ?it/s]ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab -

In [ ]:
new_manifest = PROCESSED_DATA_PATH / "manifest.jsonl"
with new_manifest.open("w", encoding="utf-8") as f:
    for entry in manifest_lines:
        entry["audio_filepath"] = entry["audio_filepath"]
        entry["samplerate"] = 16000
        entry["lang"] = "da"
        entry["source_lang"] = "da"
        entry["target_lang"] = "da"
        f.write(json.dumps(entry) + "\n")

In [51]:
TEST_SIZE = 0.2

In [52]:
from math import ceil
# Split into groups for same age and same gender
groups: dict[tuple[int, str], list] = {}

for entry in manifest_lines:
    age = int(entry["age"])
    gender = entry["gender"]
    key = (age, gender)
    if key not in groups:
        groups[key] = []
    groups[key].append(entry)

# Shuffle entries in each group
import random
for entries in groups.values():
    random.shuffle(entries)

# Print group sizes
for key, entries in groups.items():
    num_speakers = len(set(e['participant_id'] for e in entries))
    print(f"Age: {key[0]}, Gender: {key[1]}, Count: {num_speakers} speakers, {len(entries)} utterances")

    print(f"Test speakers: {ceil(num_speakers * TEST_SIZE)}")

Age: 8, Gender: F, Count: 14 speakers, 350 utterances
Test speakers: 3
Age: 8, Gender: M, Count: 15 speakers, 375 utterances
Test speakers: 3
Age: 11, Gender: F, Count: 8 speakers, 200 utterances
Test speakers: 2
Age: 11, Gender: M, Count: 7 speakers, 175 utterances
Test speakers: 2


In [53]:
test_manifest = PROCESSED_DATA_PATH / "manifest_test.jsonl"
train_manifest = PROCESSED_DATA_PATH / "manifest_train.jsonl"

with test_manifest.open("w", encoding="utf-8") as f_test, train_manifest.open("w", encoding="utf-8") as f_train:
    for key, entries in groups.items():
        participant_ids = list(set(e['participant_id'] for e in entries))
        random.shuffle(participant_ids)
        num_test_speakers = ceil(len(participant_ids) * TEST_SIZE)
        test_speakers = set(participant_ids[:num_test_speakers])
        
        for entry in entries:
            if entry['participant_id'] in test_speakers:
                f_test.write(json.dumps(entry) + "\n")
            else:
                f_train.write(json.dumps(entry) + "\n")